<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Hand Gesture Recognition (Jester) — Akida 2 Benchmark

<p align="right">Run Time: ~35 minutes</p>

Measures what the Jester TENN model costs on Akida 2 hardware: latency **per frame** in
each mapping mode, the mapping onto neural processors (NPs), and where the time goes layer
by layer.

The model is streaming: it takes one frame per inference and gives a prediction every
frame, so the latency that matters is that of one frame. There is no per-clip latency to
add up.

This notebook downloads the BrainChip Jester subset on first use, for real input frames.

> **Note:** the benchmark needs a physical Akida 2 FPGA. Everything is guarded, so the
> notebook runs end to end without one (on Colab, say), but the hardware cells are skipped.

In [ ]:
# Colab-only setup. Local users: ignore this cell - it does nothing for you.
import sys
if 'google.colab' in sys.modules:
    !wget -q https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/akida2/model_zoo/jester/colab_setup.py
import colab_setup
colab_setup.setup()

## Configuration

In [ ]:
import numpy as np
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ

DATA_PATH = None     # None: the BrainChip Jester subset, downloaded into data/
NUM_CLIPS = 10       # consecutive clips streamed by the benchmark (16 frames each)

# Clocks (see jester_benchmark.py and brainchip_utils.hardware_utils.AKIDA_CLOCKS_HZ)
MEASURED_CLOCK = AKIDA_CLOCKS_HZ['AKIDA2_FPGA']  # 25 MHz FPGA
PROJECTED_CLOCK = AKIDA_CLOCKS_HZ['AKD2500']     # 1 GHz target (pre-production)

## Model

The converted Akida model is in `pretrained_models/`, tracked with Git LFS: `verify_model`
fails if the file is a text pointer (see [Trained models](../../../README.md#trained-models)).

In [ ]:
from jester_model import model_path, verify_model

path = model_path('akida')
verify_model(path)
ak_model = akida.Model(str(path))
ak_model.summary()

## Device

`get_akida_device` returns a connected device of the model's IP version (Akida 2), or
`None` if there isn't one.

In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
if device is None:
    print('No compatible Akida hardware device found - the benchmark will be skipped.')
else:
    print('Akida device found:', device.desc)

## Samples: a stream of real frames

Akida is event-driven: computation scales with the number of non-zero activations, so
timings depend on the input and the benchmark must use real frames. For a streaming model
they must also come **in order**, so that the temporal buffers hold real history. The
samples are the frames of consecutive clips, one after the other. The buffers are not reset
between clips during timing, as in a continuous stream.

In [ ]:
from jester_data import get_data_dir, get_stream

samples = get_stream(get_data_dir(DATA_PATH), NUM_CLIPS)
print(f'{NUM_CLIPS} clips -> {samples.shape} {samples.dtype}')

## Hardware matches software

Before timing, one clip is streamed frame by frame on the hardware and on the software
backend, each from empty buffers, and the outputs compared.

In [ ]:
from jester_benchmark import check_hw_matches_sw

if device is not None:
    print('Bit-identical:', check_hw_matches_sw(ak_model, device, samples[:16]))

## Latency benchmark

`full_model_benchmark` from [brainchip_utils/hardware_utils.py](../../../brainchip_utils/hardware_utils.py)
runs one forward call per frame, so its mean latency is the latency per frame. It times the
model in each mapping mode:

- **Minimal** maps the model onto the fewest NPs.
- **AllNps** uses as many NPs as possible while keeping the minimum number of passes.
- **HwPr** (hardware partial reconfiguration) splits the model into more passes, so that
  each layer can use more NPs, and reloads the weights between passes on every inference.

The FPGA runs at **25 MHz**, so the latency **projected** to the 1 GHz AKD2500 target clock
is reported too: the cycle count of a model and mapping does not depend on the clock.

In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

full_results = {}
if device is not None:
    for mm in ['Minimal', 'AllNps', 'HwPr']:
        map_mode = getattr(akida.MapMode, mm)
        res = full_model_benchmark(ak_model, device, samples, map_mode=map_mode,
                                   batch_size=1, clock_freq=MEASURED_CLOCK)
        if res is None:
            print(f'[{mm}] does not map onto this device - skipped')
            continue
        res['projected_clk_ms'] = res['mean_inf_clk'] / PROJECTED_CLOCK * 1000
        ak_model.map(device, mode=map_mode)
        res['num_nps'], res['num_passes'], _ = get_mapping_stats(ak_model)
        full_results[mm] = res
        print(f"[{mm}] {res['num_nps']} NPs, {res['num_passes']} passes: "
              f"{res['mean_clk_ms']:.3f} ms per frame @ 25 MHz, "
              f"{res['projected_clk_ms']:.3f} ms projected @ 1 GHz")

In [ ]:
from brainchip_utils.plot_utils import plot_full_model_results

if full_results:
    ak_model.map(device, mode=akida.MapMode.Minimal)
    plot_full_model_results(full_results, ak_model, device,
                            model_name=path.name, example='Jester TENN (Akida 2)',
                            show_power=False)

## Per-layer benchmark and sparsity

`per_layer_benchmark` reconstructs the latency layer by layer, by timing cumulative
sub-models on the same frame stream and taking the differences, and `compute_sparsity`
measures each layer's activation sparsity over the stream. Sparsity is computed on the
software backend, so it is available without a device too.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.hardware_utils import per_layer_benchmark
from brainchip_utils.plot_utils import pretty_print_sparsity

sparsity_dict = compute_sparsity(ak_model, samples=samples, batch_size=len(samples))
pretty_print_sparsity(sparsity_dict)

per_layer_results = None
if full_results:
    ak_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    per_layer_results = per_layer_benchmark(ak_model, device, samples,
                                            repeats=len(samples), clock_freq=MEASURED_CLOCK)

In [ ]:
from brainchip_utils.plot_utils import plot_per_layer_results

if per_layer_results is not None:
    # Map without hw_only so the mapping plot can be drawn
    ak_model.map(device, mode=akida.MapMode.Minimal)
    plot_per_layer_results(per_layer_results, ak_model, sparsity_dict,
                           model_name=path.name, example='Jester TENN (Akida 2)')

## Summary

In [ ]:
print(f'Jester TENN, 100 x 100 frames, 8-bit; {len(samples)} streamed frames')
print(f'  mean activation sparsity: {np.mean(list(sparsity_dict.values())) * 100:.2f}%')
for mm, res in full_results.items():
    print(f"  {mm:8} {res['num_nps']:3d} NPs, {res['num_passes']:2d} passes, "
          f"{res['mean_clk_ms']:8.3f} ms per frame @ 25 MHz, "
          f"{res['projected_clk_ms']:6.3f} ms projected @ 1 GHz")